# Bringing in OULAD dataset

In [ ]:
# grab OULAD dataset
!pip install kaggle
import os
import pandas as pd

os.environ['KAGGLE_USERNAME'] = 'elisabettammjarv'
os.environ['KAGGLE_KEY'] = 'e5cbf54813421f8a937c1f9d6b49c2dd'

!kaggle datasets download -d thedevastator/open-university-learning-analytics-dataset
!unzip open-university-learning-analytics-dataset.zip

open-university-learning-analytics-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  open-university-learning-analytics-dataset.zip
replace uci-open-university-learning-analytics-dataset/OULAD.names? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
oulad_courses_df = pd.read_csv('/content/uci-open-university-learning-analytics-dataset/courses.csv')
oulad_student_assessment_df = pd.read_csv('/content/uci-open-university-learning-analytics-dataset/studentAssessment.csv')
oulad_student_info_df = pd.read_csv('/content/uci-open-university-learning-analytics-dataset/studentInfo.csv')
oulad_student_registration_df = pd.read_csv('/content/uci-open-university-learning-analytics-dataset/studentRegistration.csv')
oulad_assessments_df = pd.read_csv('/content/uci-open-university-learning-analytics-dataset/assessments.csv')

# Exploring OULAD

Here we walk through the data and explore the cohorts - it's especially interesting to see the course presentation enrollment numbers as well as what courses were taken in conjunction with each other.


In [ ]:

import pandas as pd

merged_df = pd.merge(oulad_courses_df, oulad_student_registration_df, on=['code_module', 'code_presentation'])

grouped_data = merged_df.groupby(['code_module', 'code_presentation']).size().reset_index(name='enrollment_count')

unique_years = grouped_data['code_presentation'].apply(lambda x: x[:4]).unique()
unique_semesters = grouped_data['code_presentation'].apply(lambda x: x[4:]).unique()

for year in unique_years:
    print(f"Year: {year}")
    for semester in unique_semesters:
        print(f"\tSemester: {semester}")
        filtered_data = grouped_data[(grouped_data['code_presentation'].apply(lambda x: x[:4]) == year) &
                                     (grouped_data['code_presentation'].apply(lambda x: x[4:]) == semester)]
        for index, row in filtered_data.iterrows():
            course_module = row['code_module']
            enrollment_count = row['enrollment_count']
            print(f"\t\tCourse Module: {course_module}, Enrollment Count: {enrollment_count}")




Year: 2013
	Semester: J
		Course Module: AAA, Enrollment Count: 383
		Course Module: BBB, Enrollment Count: 2237
		Course Module: DDD, Enrollment Count: 1938
		Course Module: EEE, Enrollment Count: 1052
		Course Module: FFF, Enrollment Count: 2283
		Course Module: GGG, Enrollment Count: 952
	Semester: B
		Course Module: BBB, Enrollment Count: 1767
		Course Module: DDD, Enrollment Count: 1303
		Course Module: FFF, Enrollment Count: 1614
Year: 2014
	Semester: J
		Course Module: AAA, Enrollment Count: 365
		Course Module: BBB, Enrollment Count: 2292
		Course Module: CCC, Enrollment Count: 2498
		Course Module: DDD, Enrollment Count: 1803
		Course Module: EEE, Enrollment Count: 1188
		Course Module: FFF, Enrollment Count: 2365
		Course Module: GGG, Enrollment Count: 749
	Semester: B
		Course Module: BBB, Enrollment Count: 1613
		Course Module: CCC, Enrollment Count: 1936
		Course Module: DDD, Enrollment Count: 1228
		Course Module: EEE, Enrollment Count: 694
		Course Module: FFF, Enrollmen

In [ ]:
# Initialize code_module_id starting from 1
code_module_id = 1
module_mapping = {}  # Store mapping of original code_module to new code_module_id

# Assign code_module_id for each unique combination of code_module and code_presentation
for idx, row in oulad_courses_df.iterrows():
    code_combination = (row['code_module'], row['code_presentation'])
    if code_combination not in module_mapping:
        module_mapping[code_combination] = code_module_id
        code_module_id += 1

# Update code_module_id in student info and registration dataframes
oulad_student_info_df['code_module_id'] = oulad_student_info_df.apply(
    lambda x: module_mapping[(x['code_module'], x['code_presentation'])], axis=1
)
oulad_student_registration_df['code_module_id'] = oulad_student_registration_df.apply(
    lambda x: module_mapping[(x['code_module'], x['code_presentation'])], axis=1
)

# Calculate total enrollment counts for each unique combination
enrollment_counts = oulad_student_registration_df.groupby(['code_module_id']).size().reset_index(name='enrollment_count')

# Split enrollment counts over 1300 into new code_module values
for idx, row in enrollment_counts.iterrows():
    if row['enrollment_count'] > 1300:
        new_module_id = max(enrollment_counts['code_module_id']) + 1
        module_mapping[(row['code_module_id'], 'new')] = new_module_id
        enrollment_counts.at[idx, 'code_module_id'] = new_module_id

# Update student info and registration for new code_modules
for idx, row in oulad_student_registration_df.iterrows():
    if (row['code_module_id'], 'new') in module_mapping:
        oulad_student_registration_df.at[idx, 'code_module_id'] = module_mapping[(row['code_module_id'], 'new')]
        oulad_student_info_df.at[idx, 'code_module_id'] = module_mapping[(row['code_module_id'], 'new')]


In [ ]:

# Group the assessment data by 'code_module' and 'code_presentation'
grouped_assessment = oulad_assessments_df.groupby(['code_module', 'code_presentation'])

# Iterate over each group and display the assessment breakdown
for group, data in grouped_assessment:
    code_module, code_presentation = group
    print(f"Assessment breakdown for {code_module} - {code_presentation}:")
    for idx, row in data.iterrows():
        print(f"Assessment Type: {row['assessment_type']}, Weight: {row['weight']}")
    print()


Assessment breakdown for AAA - 2013J:
Assessment Type: TMA, Weight: 10.0
Assessment Type: TMA, Weight: 20.0
Assessment Type: TMA, Weight: 20.0
Assessment Type: TMA, Weight: 20.0
Assessment Type: TMA, Weight: 30.0
Assessment Type: Exam, Weight: 100.0

Assessment breakdown for AAA - 2014J:
Assessment Type: TMA, Weight: 10.0
Assessment Type: TMA, Weight: 20.0
Assessment Type: TMA, Weight: 20.0
Assessment Type: TMA, Weight: 20.0
Assessment Type: TMA, Weight: 30.0
Assessment Type: Exam, Weight: 100.0

Assessment breakdown for BBB - 2013B:
Assessment Type: CMA, Weight: 1.0
Assessment Type: CMA, Weight: 1.0
Assessment Type: CMA, Weight: 1.0
Assessment Type: CMA, Weight: 1.0
Assessment Type: CMA, Weight: 1.0
Assessment Type: TMA, Weight: 5.0
Assessment Type: TMA, Weight: 18.0
Assessment Type: TMA, Weight: 18.0
Assessment Type: TMA, Weight: 18.0
Assessment Type: TMA, Weight: 18.0
Assessment Type: TMA, Weight: 18.0
Assessment Type: Exam, Weight: 100.0

Assessment breakdown for BBB - 2013J:
Asses

My key takeaway here is that we have 2 years worth of data for 7 courses - and the assignment breakdown seems very much more complex than one that a participant for my research would be familiar with. It also does not have any infomation about the course content, nor how they connect with each other. I will have to create my own frankenstein data set!

# Bringing in UofG realistic CompSci courses data structure


In [ ]:
# Reading in my own datastructure - this is stored at a public address
import pandas as pd
import requests

r = requests.get('https://docs.google.com/spreadsheets/d/e/2PACX-1vSNqV2dHTyyjYZxSBWmE8narOxIGqBTtbTL2X6hkT92uBeC20LbD7QDikel483vaaLRGvnLqa39czcA/pub?output=xlsx')
open('real_courses_no_learners_dataset.xslx', 'wb').write(r.content)
my_assessment_df = pd.read_excel('real_courses_no_learners_dataset.xslx', sheet_name='Assessment', usecols=lambda x: 'Unnamed' not in x)
my_assessment_course_df = pd.read_excel('real_courses_no_learners_dataset.xslx', sheet_name='Assessment-Course', usecols=lambda x: 'Unnamed' not in x)
my_course_df =pd.read_excel('real_courses_no_learners_dataset.xslx', sheet_name='Course', usecols=lambda x: 'Unnamed' not in x)
my_course_learner_df =pd.read_excel('real_courses_no_learners_dataset.xslx', sheet_name='Course-Learner', usecols=lambda x: 'Unnamed' not in x)
my_learner_df = pd.read_excel('real_courses_no_learners_dataset.xslx', sheet_name='Learner', usecols=lambda x: 'Unnamed' not in x)
my_learner_assessment_df = pd.read_excel('real_courses_no_learners_dataset.xslx', sheet_name='Learner-Assessment', usecols=lambda x: 'Unnamed' not in x)


# Reconstructing the OULAD with UofG realistic CompSci course data



Here we modify the OULAD dataset to fit my original blank data structure. This is done so my participants can see a more accurate representation of the LA dashboard that is in line with their mental schema surorunfing university structure - simplifying the mental load for them, and letting me get to asking questions about the parts that matter.


In [ ]:
#replace code_module with id
code_module_mapping = {code: idx+1 for idx, code in enumerate(oulad_courses_df['code_module'].unique())}
oulad_student_info_df['code_module'] = oulad_student_info_df['code_module'].map(code_module_mapping)
oulad_assessments_df['code_module'] = oulad_assessments_df['code_module'].map(code_module_mapping)

#save full ones for synthesizing later
full_oulad_student_assessments_df = oulad_student_assessment_df.copy()
full_oulad_assessments_df = oulad_assessments_df.copy()

#remove excess assessments from oulad_assessments_df
max_assessments = my_assessment_course_df.groupby('Course ID')['Assessment ID'].count()
assessments_in_courses = oulad_assessments_df.groupby('code_module')

limited_assessments = {}

for course, assessments in assessments_in_courses:
    max_count = max_assessments.get(course, 0)
    limited_assessments[course] = assessments.head(max_count)

limited_assessments_list = [value for value in limited_assessments.values()]
limited_assessments_df = pd.concat(limited_assessments_list)

oulad_assessments_df = limited_assessments_df

#remove excess assessments from oulad_student_assessment_df
assessments_in_oulad = oulad_assessments_df['id_assessment'].unique()
oulad_student_assessment_df = oulad_student_assessment_df[oulad_student_assessment_df['id_assessment'].isin(assessments_in_oulad)]

#remap the assessment_ids
assessment_id_mapping = {code: idx+1 for idx, code in enumerate(assessments_in_oulad)}
oulad_student_assessment_df["id_assessment"] = oulad_student_assessment_df["id_assessment"].map(assessment_id_mapping)
oulad_assessments_df["id_assessment"] = oulad_assessments_df["id_assessment"].map(assessment_id_mapping)


#chuck real course data in my dataframes
my_assessment_df.loc[0:len(oulad_assessments_df["id_assessment"].unique()) - 1, "Assessment ID"] = oulad_assessments_df["id_assessment"].unique()
my_assessment_df.dropna(subset=["Assessment ID"], inplace=True)

my_assessment_course_df.loc[0:len(oulad_assessments_df["id_assessment"])-1, "Assessment ID"] = oulad_assessments_df["id_assessment"].reset_index(drop=True)

my_course_learner_df["Learner ID"] = oulad_student_info_df["id_student"]
my_course_learner_df["Course ID"] = oulad_student_info_df["code_module"]
my_course_learner_df.dropna(subset=["Course ID", "Learner ID"], inplace=True)

my_learner_df["Learner ID"] = oulad_student_info_df["id_student"]
my_learner_df.dropna(subset=["Learner ID"], inplace=True)

my_learner_assessment_df['Learner ID'] = oulad_student_assessment_df["id_student"]
my_learner_assessment_df['Assessment ID'] = oulad_student_assessment_df["id_assessment"]
my_learner_assessment_df['Assessment Grade'] = oulad_student_assessment_df["score"]
my_learner_assessment_df.dropna(subset=["Assessment ID", "Learner ID", 'Assessment Grade'], inplace=True)


<ipython-input-7-de01dfcb5457>:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  oulad_student_assessment_df["id_assessment"] = oulad_student_assessment_df["id_assessment"].map(assessment_id_mapping)


# Synthesizing data for the other 29 courses


In [ ]:
%pip install sdv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 27.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 17.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.6 MB/s eta 0:00:00


In [ ]:
data_for_synthesizer = pd.merge(full_oulad_student_assessments_df, full_oulad_assessments_df, on='id_assessment', how='left')


In [ ]:
from sdv.metadata import SingleTableMetadata
from sdv.lite import SingleTablePreset

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data_for_synthesizer)

from sdv.single_table import GaussianCopulaSynthesizer
synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(data_for_synthesizer)
synthetic_data = synthesizer.sample(num_rows=8100)
synthetic_data.head()

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight
0,36435,841507,63,0,0,7,2014J,TMA,222,0.1
1,35401,583361,11,0,75,6,2014B,TMA,23,0.1
2,24958,957229,178,0,91,2,2013J,TMA,214,8.9
3,19024,1424793,66,0,100,2,2014J,TMA,187,2.5
4,26696,200697,175,0,86,3,2014J,Exam,152,0.0


In [ ]:
# I had hoped to be able to create several fake courses, but sdv limits itself to sticking with my og data too much
# And so it only creates 6 courses
# And chopping those up to create 30 other courses is illogical and time-consuming
# So the rest of my courses will have to just be random scores
# The end product focuses more on the aggregation of cohort scores and I already have some real data to display
# Individual analytics from, so I think this is the best approach
# synthetic_data contains id_assessment	id_student	date_submitted	is_banked	score	code_module	code_presentation	assessment_type	date	weight

import random
from random import randint, randrange
random.seed(0)
import numpy as np

the_rest_of_ass = my_assessment_course_df[my_assessment_course_df['Course ID']>7]["Assessment ID"]
synthetic_data_list = synthetic_data.values.tolist()
kidnapped_learners = oulad_student_info_df["id_student"].unique().tolist()

new_assessments = {
    'Learner ID': [],
    'Assessment ID': [],
    'Learner-Assessment': [],
    'Assessment Grade': [],
    'Completed': []
}

new_courses = {
    'Course ID': [],
    'Learner ID': [],
    'Course-Learner': [],
    'Course Grade': []
}

# 8100 records / 81 assessments, so 100 times
for i in range(100):
    student = random.sample(kidnapped_learners, 1)[0]
    for assessment in the_rest_of_ass:
        sample = random.sample(synthetic_data_list, 1)[0]
        new_assessments['Learner ID'].append(student)
        new_assessments['Assessment ID'].append(assessment)
        new_assessments['Learner-Assessment'].append("")
        new_assessments['Assessment Grade'].append(sample[4])
        new_assessments['Completed'].append("")
    for course in range(7, 36):
        new_courses['Course ID'].append(course)
        new_courses['Learner ID'].append(student)
        new_courses['Course-Learner'].append("")
        new_courses['Course Grade'].append("")

my_learner_assessment_df = pd.concat([my_learner_assessment_df, pd.DataFrame(new_assessments)], ignore_index=True)
my_course_learner_df = pd.concat([my_course_learner_df, pd.DataFrame(new_courses)], ignore_index=True)


# Writing Data back to an excel sheet for the backend <3

In [ ]:
!pip install xlsxwriter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.7 MB/s eta 0:00:00


In [ ]:
import xlsxwriter
final = {
   "Assessment" : my_assessment_df,
   "Assessment-Course" : my_assessment_course_df,
   "Course" : my_course_df,
   "Course-Learner": my_course_learner_df,
   "Learner": my_learner_df,
   "Learner-Assessment": my_learner_assessment_df
}


writer = pd.ExcelWriter('Real Courses Real Learners.xlsx', engine='xlsxwriter')
for sheet_name in final:
    final[sheet_name].to_excel(writer, sheet_name= sheet_name, index=False)

writer.save()

<ipython-input-18-a2cfeda5ea71>:16: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  writer.save()
